# 1. 단일 기업 재무제표 받기

In [179]:
# 1. 단일 기업 재무제표 수집 + WRDS 보완
import sys
import pandas as pd
import numpy as np
import pymysql

# 경로 추가
sys.path.append(r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\collect")

# 필수 모듈 import
from sec_data_pipeline.collectors.sec_utils import fetch_company_facts
from sec_data_pipeline.parsers.company_facts_parser import CompanyFactsParser
from sec_data_pipeline.parsers.financial_normalizer import FinancialNormalizer
from sec_data_pipeline.validators.wrds_data_validator_mysql_integrated import WRDSDataValidator
from DATA.stock_invest_function import get_db_host
from sec_data_pipeline.validators.edgar_wrds_filler import (
    fetch_wrds_fundq_sample, fill_revenue_with_wrds
)

TICKER = "ADBE"   # ← 여기만 바꾸면 전체가 따라옵니

# 1) EDGAR 수집/정규화
headers = {"User-Agent": "HoyoungPark Research <stox1224@email.com>"}
facts   = fetch_company_facts( TICKER, headers=headers)
parser  = CompanyFactsParser(facts)
normal  = FinancialNormalizer(parser)
edgar_df = normal.create_normalized_dataframe(period_type="quarterly")

# 2) WRDS Validator
db_info = {
    "host": get_db_host(),
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
validator = WRDSDataValidator(db_info)

# 3) 결합/보완 실행 (핵심: table_name='us_fundq', date_col='edate')
result_df = validator.validate_and_fill_improved(
    edgar_df=edgar_df,
    ticker= TICKER,
    table_name="us_fundq",   # 당신 DB 테이블
    days_tolerance=15,
    verbose=True,
    ticker_col="ticker",     # 당신 DB 컬럼
    date_col="edate"         # 당신 DB 컬럼
)


import sqlalchemy as sa

# ① EDGAR 분기 DF 준비 (이미 갖고 계신 df: edgar_df)
#    edgar_df.index = 분기 날짜, 컬럼에 'revenue' 포함

# ② WRDS(us_fundq)에서 AAPL의 edate, ticker, saleq 로드
db_info = {
    "host": "192.168.0.230",
    "port": 3307,
    "user": "stox7412",
    "password": "Apt106503!~",
    "database": "investar",
}
df_wrds = fetch_wrds_fundq_sample(
    db_info, ticker=TICKER , columns=["edate", "ticker", "saleq"], table_name="US_fundq"
)

# ③ 결측 보완 + 단위 정합
filled_df = fill_revenue_with_wrds(
    edgar_df=edgar_df,
    wrds_df=df_wrds,
    wrds_value_col="saleq",     # WRDS의 분기 매출 컬럼명
    days_tolerance=20,          # 분기말 근접 허용일
    verbose=True
)

print("\n[CHECK] revenue NaN:")
print("  before:", edgar_df["revenue"].isna().sum())
print("  after :", filled_df["revenue"].isna().sum())

if filled_df.empty:
    print("⚠ filled_df가 비어 있습니다 → result_df를 target_df로 대체합니다.")
    target_df = result_df.copy()
else:
    print("✓ filled_df가 유효합니다 → target_df에 filled_df를 사용합니다.")
    target_df = filled_df.copy()

✓ MySQL 연결 성공: 192.168.0.230:3307/investar
⚠ WRDS 쿼리 실패: Execution failed on sql 'SELECT * FROM us_fundq WHERE ticker=%s': (1146, "Table 'investar.us_fundq' doesn't exist")
WRDS 데이터 없음 → EDGAR 그대로 반환
[fill] pattern=(3, 6, 9, 12), scale_factor=1.0, filled=0

[CHECK] revenue NaN:
  before: 2
  after : 0
⚠ filled_df가 비어 있습니다 → result_df를 target_df로 대체합니다.


In [180]:
target_df

,revenue,net_income,operating_income,gross_profit,total_assets,current_assets,total_liabilities,current_liabilities,stockholders_equity,cash,...,other_noncurrent_assets,short_term_debt,long_term_debt_current,total_debt,accounts_payable,accounts_receivable,deferred_revenue,accrued_liabilities,depreciation_amortization,capital_expenditures
date,,,,,,,,,,,,,,,,,,,,,
2006-12-01,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,5.151876e+09,7.725000e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2007-11-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,4.649982e+09,9.464220e+08,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-05-30,8.868860e+08,2.149100e+08,2.601780e+08,8.040200e+08,NaN,NaN,NaN,NaN,4.649982e+09,1.162453e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-08-29,8.872570e+08,1.916080e+08,2.194690e+08,7.764060e+08,NaN,NaN,NaN,NaN,4.649982e+09,1.134263e+09,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2008-11-28,8.872570e+08,1.916080e+08,2.194690e+08,7.764060e+08,5.821598e+09,2.735103e+09,1.411244e+09,7.625990e+08,4.410354e+09,8.864500e+08,...,2.165290e+08,NaN,NaN,NaN,55840000.0,4.672340e+08,2.439640e+08,3.999690e+08,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-30,5.408000e+09,1.684000e+09,1.992000e+09,4.854000e+09,2.983000e+10,1.071600e+10,1.528500e+10,9.644000e+09,1.454500e+10,7.193000e+09,...,1.557000e+09,1.499000e+09,NaN,1.907231e+09,318000000.0,1.802000e+09,3.317000e+09,1.848000e+09,212000000.0,37000000.0
2024-11-29,5.408000e+09,1.684000e+09,1.992000e+09,4.854000e+09,3.023000e+10,1.123200e+10,1.612500e+10,1.052100e+10,1.410500e+10,7.613000e+09,...,1.554000e+09,1.499000e+09,1.500000e+09,1.907231e+09,361000000.0,2.072000e+09,3.317000e+09,2.336000e+09,212000000.0,37000000.0
2025-02-28,5.714000e+09,1.811000e+09,2.163000e+09,5.092000e+09,2.995500e+10,1.085500e+10,1.686000e+10,9.163000e+09,1.309500e+10,6.758000e+09,...,1.638000e+09,0.000000e+00,1.500000e+09,1.907231e+09,326000000.0,1.973000e+09,3.317000e+09,1.951000e+09,217000000.0,26000000.0


In [181]:
from sec_data_pipeline.parsers.financial_normalizer import FinancialNormalizer


# parser는 EDGAR 데이터를 읽어온 CompanyFactsParser 등의 인스턴스
normalizer = FinancialNormalizer(parser=None)  # parser가 필요 없으면 None으로 두세요


# 1) 기본 추정: income_tax_expense / pretax_income
est = (filled_df.get('income_tax_expense') / filled_df.get('pretax_income'))

# 2) 이상치/음수/무한대 정리
est = est.replace([np.inf, -np.inf], np.nan)
# 적자 구간(pretax_income <= 0)은 추정치 제거
est = est.mask((filled_df.get('pretax_income') <= 0), np.nan)
# 0~0.5로 클리핑(50% 이상은 보통 이상치)
est = est.clip(lower=0.0, upper=0.3)

# 3) 결측 보강: 최근 값으로 보간 + 기본값 대체
# est = est.ffill().fillna(0.21)

# filled_df['tax_rate'] = 0.21
ratio = normal.calculate_financial_ratios(target_df)

# # 예: AAPL에 대해 2018~현재 21%, 그 이전 35%
# filled_df['tax_rate'] = np.where(filled_df.index < '2018-01-01', 0.35, 0.21)
# ratio = normal.calculate_financial_ratios(filled_df)

print(ratio.tail())

                 revenue    net_income  operating_income  gross_profit  \
date                                                                     
2024-08-30  5.408000e+09  1.684000e+09      1.992000e+09  4.854000e+09   
2024-11-29  5.408000e+09  1.684000e+09      1.992000e+09  4.854000e+09   
2025-02-28  5.714000e+09  1.811000e+09      2.163000e+09  5.092000e+09   
2025-05-30  5.873000e+09  1.691000e+09      2.109000e+09  5.235000e+09   
2025-08-29  5.988000e+09  1.772000e+09      2.173000e+09  5.346000e+09   

            total_assets  current_assets  total_liabilities  \
date                                                          
2024-08-30  2.983000e+10    1.071600e+10       1.528500e+10   
2024-11-29  3.023000e+10    1.123200e+10       1.612500e+10   
2025-02-28  2.995500e+10    1.085500e+10       1.686000e+10   
2025-05-30  2.810700e+10    8.978000e+09       1.665900e+10   
2025-08-29  2.875400e+10    9.412000e+09       1.698400e+10   

            current_liabilities  stockh

In [182]:
ratio[["roic_standard", "roic_capital_structure",  "roic_working_capital"]]

,roic_standard,roic_capital_structure,roic_working_capital
date,,,
2006-12-01,NaN,NaN,NaN
2007-11-30,NaN,NaN,NaN
2008-05-30,NaN,5.863765,NaN
2008-08-29,NaN,4.906814,NaN
2008-11-28,3.427170,4.895474,NaN
...,...,...,...
2024-08-30,7.575586,12.552501,-61.921067
2024-11-29,7.745973,13.434285,-47.382922
2025-02-28,8.634237,14.240764,-56.578334


In [2]:
import os

collectors_path = r"C:\Users\82108\OneDrive\바탕 화면\investment\investment_strategy\US_Market\collect\sec_data_pipeline\collectors"

print("collectors 폴더 내용:")
for f in os.listdir(collectors_path):
    print(f"  {f}")

# sec_api_client.py 파일이 있는지 확인
sec_api_file = os.path.join(collectors_path, "sec_api_client.py")
print(f"\nsec_api_client.py 존재: {os.path.exists(sec_api_file)}")

collectors 폴더 내용:
  aapl_company_facts.json
  bulk_downloader.py
  rate_limiter.py
  sec_api_client.py
  sec_utils.py
  __init__.py
  __pycache__

sec_api_client.py 존재: True


In [5]:
import pandas as pd
import pymysql
from DATA.stock_invest_function import get_db_host

# DB 연결
conn = pymysql.connect(
    host=get_db_host(),
    port=3307,
    user='stox7412',
    password='Apt106503!~',
    database='investar',
    charset='utf8mb4'
)

# 1. 테이블 구조 확인
print("=" * 80)
print("US_fundq 테이블 구조 확인")
print("=" * 80)

query_structure = "SHOW COLUMNS FROM US_fundq"
columns_df = pd.read_sql(query_structure, conn)

print("\n모든 컬럼 목록:")
print(columns_df[['Field', 'Type']].to_string(index=False))

# 2. 날짜 관련 컬럼 찾기
date_columns = columns_df[columns_df['Field'].str.contains('date|time|year|quarter', case=False, na=False)]
print(f"\n날짜 관련 컬럼 ({len(date_columns)}개):")
print(date_columns['Field'].tolist())

# 3. Revenue 관련 컬럼 찾기
revenue_columns = columns_df[columns_df['Field'].str.contains('sale|rev|income', case=False, na=False)]
print(f"\nRevenue/Income 관련 컬럼 ({len(revenue_columns)}개):")
print(revenue_columns['Field'].tolist())

# 4. 데이터 샘플 조회 (날짜 컬럼 없이)
print("\n" + "=" * 80)
print("데이터 샘플 조회")
print("=" * 80)

query_sample = """
SELECT *
FROM US_fundq
WHERE tic = 'AAPL'
LIMIT 5
"""

df = pd.read_sql(query_sample, conn)
print(f"\n조회된 데이터: {len(df)} rows × {len(df.columns)} columns")

# 5. 주요 컬럼만 출력
if not df.empty:
    print("\n샘플 데이터 (주요 컬럼):")

    # 출력할 컬럼 선택
    display_cols = []

    # ticker 컬럼
    if 'tic' in df.columns:
        display_cols.append('tic')

    # 날짜 컬럼
    for date_col in date_columns['Field'].tolist():
        if date_col in df.columns:
            display_cols.append(date_col)
            break

    # revenue 관련
    for rev_col in revenue_columns['Field'].tolist()[:3]:
        if rev_col in df.columns:
            display_cols.append(rev_col)

    if display_cols:
        print(df[display_cols].head().to_string(index=False))
    else:
        print(df.head())

conn.close()

print("\n" + "=" * 80)
print("확인 완료")
print("=" * 80)

US_fundq 테이블 구조 확인

모든 컬럼 목록:
    Field       Type
   permno bigint(20)
    edate       text
     date   datetime
    dlret     double
   dlretx     double
   exchcd bigint(20)
    naics bigint(20)
   permco bigint(20)
      prc     double
      ret     double
    shrcd bigint(20)
   shrout     double
    siccd     double
   ticker       text
    cusip       text
 rankyear bigint(20)
   retadj     double
       me     double
Adj Close     double
    gvkey     double
      tic       text
     actq     double
      aoq     double
      apq     double
      atq     double
     ceqq     double
     cheq     double
    cogsq     double
     dlcq     double
      dpq     double
   epspiq     double
      ibq     double
   icaptq     double
   intanq     double
    invtq     double
    ivaoq     double
    ivstq     double
     lctq     double
      loq     double
      ltq     double
    mibtq     double
      niq     double
      piq     double
   ppegtq     double
   ppentq     double
   p

### 분기별 vs 연간 데이터분기별

In [3]:
# 분기별 데이터
df_quarterly = normalizer.create_normalized_dataframe(period_type='quarterly')

# 연간 데이터
df_annual = normalizer.create_normalized_dataframe(period_type='annual')

In [2]:
df_quarterly

NameError: name 'df_quarterly' is not defined

In [13]:
df_quarterly

,revenue,net_income,operating_income,gross_profit,total_assets,current_assets,total_liabilities,current_liabilities,stockholders_equity,cash,long_term_debt,cost_of_revenue,operating_expenses,research_development
date,,,,,,,,,,,,,,
2006-09-30,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,9.984000e+09,6392000000,NaN,NaN,NaN,NaN
2007-09-29,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.453100e+10,9352000000,NaN,NaN,NaN,NaN
2008-06-28,NaN,3.698000e+09,4.833000e+09,8.406000e+09,NaN,NaN,NaN,NaN,NaN,9373000000,NaN,1.617800e+10,3.573000e+09,8.110000e+08
2008-09-27,NaN,NaN,NaN,NaN,3.617100e+10,3.000600e+10,1.387400e+10,1.136100e+10,2.229700e+10,11875000000,NaN,NaN,NaN,NaN
2008-12-27,NaN,2.255000e+09,3.101000e+09,4.507000e+09,NaN,NaN,NaN,NaN,NaN,7236000000,NaN,7.373000e+09,1.406000e+09,3.150000e+08
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-09-28,NaN,NaN,NaN,NaN,3.649800e+11,1.529870e+11,3.080300e+11,1.763920e+11,5.695000e+10,29943000000,9.666200e+10,NaN,NaN,NaN
2024-12-28,NaN,3.633000e+10,4.283200e+10,5.827500e+10,3.440850e+11,1.332400e+11,2.773270e+11,1.443650e+11,6.675800e+10,30299000000,9.480000e+10,6.602500e+10,1.544300e+10,8.268000e+09
2025-03-29,NaN,6.111000e+10,7.242100e+10,1.031420e+11,3.312330e+11,1.186740e+11,2.644370e+11,1.445710e+11,6.679600e+10,28162000000,9.220000e+10,1.165170e+11,3.072100e+10,1.681800e+10


### Billions 단위로 변환

In [ ]:
df_billions = normalizer.convert_to_billions(df)
print(df_billions[['revenue', 'net_income']].tail(8))

### TTM (Trailing Twelve Months) 계산

In [ ]:
df_ttm = normalizer.calculate_ttm(df_billions, columns=['revenue', 'net_income'])
print(df_ttm[['revenue', 'revenue_ttm']].tail(8))

# 2. 다수 기업 재무제표 받기

In [ ]:
from collectors import SECAPIClient, RateLimiter, BulkDownloader
from parsers import CompanyFactsParser, FinancialNormalizer
import pandas as pd

# 1. 클라이언트 및 다운로더 설정
user_agent = "PersonalResearch stox1224@email.com"
client = SECAPIClient(user_agent, RateLimiter(10, 1.0))
downloader = BulkDownloader(client, output_dir="./sec_data", max_workers=3)

# 2. 다운로드할 기업 리스트
tickers = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'TSLA', 'NVDA', 'META']

# 3. 배치 다운로드 (자동으로 JSON 파일 저장)
results = downloader.download_company_facts_batch(tickers)

# 4. 각 기업별 데이터 처리
all_financials = {}

for ticker, company_facts in results.items():
    parser = CompanyFactsParser(company_facts)
    normalizer = FinancialNormalizer(parser)

    # 재무데이터 정규화
    df = normalizer.create_normalized_dataframe(period_type='quarterly')
    df_billions = normalizer.convert_to_billions(df)

    all_financials[ticker] = df_billions

    # 개별 CSV 저장
    normalizer.export_to_csv(df_billions, f'{ticker}_financials.csv')

print(f"총 {len(all_financials)}개 기업 데이터 수집 완료")

# 3. 특정 재무항목만 추출

In [ ]:
revenue_comparison = {}

for ticker, company_facts in results.items():
    parser = CompanyFactsParser(company_facts)
    normalizer = FinancialNormalizer(parser)

    # Revenue만 추출
    revenue_series = normalizer.normalize_single_item('revenue', period_type='quarterly')

    if revenue_series is not None:
        revenue_comparison[ticker] = revenue_series / 1_000_000_000  # Billions

# DataFrame으로 변환
revenue_df = pd.DataFrame(revenue_comparison)
print("\n최근 8분기 매출 비교 (Billions):")
print(revenue_df.tail(8))

# YoY 성장률
growth_df = revenue_df.pct_change(periods=4) * 100
print("\nYoY 매출 성장률 (%):")
print(growth_df.tail(4))

# 4. 사용 가능한 재무 항목


In [ ]:
# 추출 가능한 표준 재무 항목들
available_items = [
    'revenue',              # 매출
    'net_income',           # 순이익
    'operating_income',     # 영업이익
    'gross_profit',         # 매출총이익
    'total_assets',         # 총자산
    'current_assets',       # 유동자산
    'total_liabilities',    # 총부채
    'current_liabilities',  # 유동부채
    'stockholders_equity',  # 자본
    'cash',                 # 현금
    'long_term_debt',       # 장기부채
    'cost_of_revenue',      # 매출원가
    'operating_expenses',   # 영업비용
    'research_development', # 연구개발비
    'earnings_per_share',   # 주당순이익
]

# 특정 항목만 추출
for item in ['revenue', 'net_income', 'total_assets']:
    series = normalizer.normalize_single_item(item, period_type='quarterly')
    print(f"\n{item}:")
    print(series.tail(4))

# 5. 재무비율 계산

In [ ]:
# 데이터 준비
df = normalizer.create_normalized_dataframe(period_type='quarterly')

# 재무비율 자동 계산
df_ratios = normalizer.calculate_financial_ratios(df)

# 계산되는 비율들:
# - profit_margin: 순이익률
# - operating_margin: 영업이익률
# - current_ratio: 유동비율
# - debt_to_equity: 부채비율
# - roe: 자기자본이익률
# - roa: 총자산이익률

print(df_ratios[['revenue', 'net_income', 'profit_margin']].tail(8))

# 6. 성장률 분석

In [ ]:
df_billions = normalizer.convert_to_billions(df)

# YoY 성장률 계산
df_growth = normalizer.calculate_growth_rates(
    df_billions,
    columns=['revenue', 'net_income'],
    periods=4  # 4분기 = YoY
)

print(df_growth[['revenue', 'revenue_yoy_growth']].tail(8))

# 7. 진행상황 모니터링

In [ ]:
# 콜백 함수 정의
def on_download_complete(ticker, data, index, total):
    if data:
        entity_name = data.get('entityName', 'Unknown')
        print(f"[{index}/{total}] {ticker}: {entity_name} - 완료")
    else:
        print(f"[{index}/{total}] {ticker}: 실패")

# 콜백과 함께 다운로드
results = downloader.download_company_facts_batch(
    tickers=['AAPL', 'MSFT', 'GOOGL'],
    callback=on_download_complete
)

# 8. 중단된 다운로드 재개


In [ ]:
# 이미 다운로드된 파일은 건너뛰기
remaining_results = downloader.resume_download(tickers)

# 9. 다운로드 리포트 생성

In [ ]:
# 다운로드 후 리포트 생성
downloader.create_download_report(results, "download_report.json")

# 10. 실전 예제: S&P 500 상위 기업 분석

In [ ]:
# Tech Giants
tech_giants = ['AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META', 'NVDA', 'TSLA']

# 다운로드
downloader = BulkDownloader(client, output_dir="./tech_giants", max_workers=5)
results = downloader.download_company_facts_batch(tech_giants)

# Revenue 비교 분석
revenue_data = {}
for ticker, facts in results.items():
    parser = CompanyFactsParser(facts)
    normalizer = FinancialNormalizer(parser)
    revenue = normalizer.normalize_single_item('revenue', period_type='quarterly')
    if revenue is not None:
        revenue_data[ticker] = revenue / 1e9  # Billions

# 비교 DataFrame
comparison = pd.DataFrame(revenue_data)
print("\nTech Giants 최근 매출 비교 (Billions):")
print(comparison.tail(8))

# 시각화 (선택사항)
import matplotlib.pyplot as plt
comparison.tail(12).plot(figsize=(12, 6))
plt.title('Tech Giants Revenue Comparison (Last 12 Quarters)')
plt.ylabel('Revenue (Billions USD)')
plt.legend(loc='best')
plt.grid(True)
plt.savefig('tech_giants_revenue.png')